# Chapter 10 live coding — Fashion MNIST MLP in PyTorch

**Reference:** Aurélien Géron, *Hands-On Machine Learning with Scikit-Learn and PyTorch*, Ch. 10, “Building an Image Classifier with PyTorch”.

> Classroom adaptation: the architecture follows the chapter's Fashion-MNIST MLP (`784 → 300 → 100 → 10`). `FAST_MODE=True` uses a smaller training subset so the live demo fits in ~30 minutes; set it to `False` for the full 55k/5k split used in the book.


In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
import torchvision
import torchvision.transforms.v2 as T
import matplotlib.pyplot as plt

torch.manual_seed(42)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print("device:", device)


## 0. Data — already prepared

Géron uses Fashion MNIST through TorchVision. Images arrive as `[C, H, W] = [1, 28, 28]`, while a mini-batch has shape `[B, 1, 28, 28]`.

The preprocessing below converts images to `float32` tensors and scales pixels to `[0, 1]`.


In [ ]:
to_tensor = T.Compose([
    T.ToImage(),
    T.ToDtype(torch.float32, scale=True),
])

train_and_valid_data = torchvision.datasets.FashionMNIST(
    root="datasets", train=True, download=True, transform=to_tensor
)
test_data = torchvision.datasets.FashionMNIST(
    root="datasets", train=False, download=True, transform=to_tensor
)

generator = torch.Generator().manual_seed(42)
train_data, valid_data = torch.utils.data.random_split(
    train_and_valid_data, [55_000, 5_000], generator=generator
)

# Classroom shortcut: keep the same task/model, reduce only the amount of data.
FAST_MODE = True
if FAST_MODE:
    train_data = Subset(train_data, range(12_000))
    valid_data = Subset(valid_data, range(2_000))

BATCH_SIZE = 128 if FAST_MODE else 32
pin = device.type == "cuda"

train_loader = DataLoader(
    train_data, batch_size=BATCH_SIZE, shuffle=True, pin_memory=pin
)
valid_loader = DataLoader(
    valid_data, batch_size=BATCH_SIZE, shuffle=False, pin_memory=pin
)
test_loader = DataLoader(
    test_data, batch_size=BATCH_SIZE, shuffle=False, pin_memory=pin
)

class_names = train_and_valid_data.classes
print(f"train={len(train_data)}, valid={len(valid_data)}, test={len(test_data)}")
print(class_names)


## 1. Inspect one mini-batch — reason before coding

Before defining the network, predict the shapes:

- `X_batch.shape = ?`
- `y_batch.shape = ?`
- after `nn.Flatten()`, each image has how many features?
- how many output units do we need?

The dataset loading is done; this shape reasoning is worth doing live.


In [ ]:
X_batch, y_batch = next(iter(train_loader))
print("X_batch:", X_batch.shape, X_batch.dtype)
print("y_batch:", y_batch.shape, y_batch.dtype)
print("first labels:", y_batch[:8].tolist())

fig, axes = plt.subplots(1, 6, figsize=(10, 2))
for ax, image, label in zip(axes, X_batch[:6], y_batch[:6]):
    ax.imshow(image.squeeze(0), cmap="gray")
    ax.set_title(class_names[label])
    ax.axis("off")
plt.tight_layout()


## 2. Build the classifier — **LIVE**

Géron's classifier is a custom `nn.Module` containing an `nn.Sequential` stack:

\[
[1,28,28] \rightarrow 784 \rightarrow 300 \rightarrow 100 \rightarrow 10.
\]

Questions to ask while filling it in:

- Why is `Flatten` needed?
- Where do ReLUs go?
- Why are there **10** outputs?
- Why is there **no Softmax** in the final layer?


In [ ]:
class ImageClassifier(nn.Module):
    def __init__(self, n_inputs, n_hidden1, n_hidden2, n_classes):
        super().__init__()

        # TODO 1 — build the MLP
        # self.mlp = nn.Sequential(
        #     ...
        # )

    def forward(self, X):
        # TODO 2 — define the forward pass
        pass


torch.manual_seed(42)

# TODO 3 — instantiate the model and move it to `device`
# model = ...

model


## 3. Loss + one forward pass — **LIVE**

For exclusive multiclass classification, Géron uses:

```python
nn.CrossEntropyLoss()
```

Important: it expects **logits**, so we do **not** add a Softmax layer to the model.

Predict first:

- shape of the logits for a batch of size `B`;
- what `argmax(dim=1)` returns;
- why the targets can remain integer class indices.


In [ ]:
# TODO 4 — choose the loss
# criterion = ...

X_batch, y_batch = next(iter(train_loader))
X_batch = X_batch.to(device)
y_batch = y_batch.to(device)

# TODO 5 — forward pass and loss
# logits = ...
# loss = ...

# print("logits shape:", logits.shape)
# print("loss:", loss.item())


## 4. Optimizer — **LIVE**

The optimizer owns the parameter-update rule. Create it **after** moving the model to the accelerator.

For a short live demo we use SGD, matching the Chapter 10 discussion.

> The learning rate below is chosen for a short classroom run, not as a claim of an optimal hyperparameter.


In [ ]:
# TODO 6 — create the optimizer
# optimizer = torch.optim.SGD(...)


## 5. One training step — **LIVE, line by line**

This is the most important cell of the notebook.

Put these operations in the correct order:

1. forward pass;
2. compute loss;
3. backpropagation;
4. update parameters;
5. clear accumulated gradients.

Then inspect whether the loss is a scalar and whether gradients appeared.


In [ ]:
X_batch, y_batch = next(iter(train_loader))
X_batch, y_batch = X_batch.to(device), y_batch.to(device)

# TODO 7 — complete ONE optimization step
# logits = ...
# loss = ...
# ...
# ...
# ...

print("loss:", float(loss))
first_weight = next(model.parameters())
print("gradient available:", first_weight.grad is not None)


## 6. Mini-batch training loop — **LIVE**

Now generalize the single step to all batches and epochs.

The essential PyTorch pattern is:

```text
model.train()
for batch:
    forward → loss → backward → step → zero_grad
```

The helper for validation is intentionally already provided below.


In [ ]:
def train_one_epoch(model, loader, criterion, optimizer):
    # TODO 8 — switch to training mode
    # ...

    running_loss = 0.0

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        # TODO 9 — the five essential training operations
        # ...
        # ...
        # ...
        # ...
        # ...

        running_loss += loss.item()

    return running_loss / len(loader)


## 7. Validation — helper already prepared

Evaluation is secondary to today's learning objective, so this function is given.

Notice the two important differences from training:

- `model.eval()`;
- no gradient tracking.

Accuracy is used here as an **evaluation metric**, not as the differentiable training loss.


In [ ]:
@torch.no_grad()
def evaluate_accuracy(model, loader):
    model.eval()
    correct = 0
    total = 0

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        logits = model(X_batch)
        predictions = logits.argmax(dim=1)

        correct += (predictions == y_batch).sum().item()
        total += y_batch.numel()

    return correct / total


## 8. Train briefly — **LIVE**

For the live session, 2–3 epochs are enough to observe learning.

With `FAST_MODE=False`, you can reproduce the full dataset split and train longer after class.


In [ ]:
N_EPOCHS = 3 if FAST_MODE else 5

# TODO 10 — call the training function for each epoch
# for epoch in range(N_EPOCHS):
#     ...
#     ...
#     print(...)

# Expected qualitative behavior:
# - training loss should decrease;
# - validation accuracy should rise well above chance (10%).


## 9. From logits to predictions and probabilities — **LIVE if time**

During training we feed **logits** directly to `CrossEntropyLoss`.

At inference:

- class prediction: `argmax(logits)`;
- probabilities: `softmax(logits)`.

This is exactly why Softmax does not belong inside this classifier's output layer during training.


In [ ]:
model.eval()
X_new, y_new = next(iter(valid_loader))
X_new = X_new[:6].to(device)
y_new = y_new[:6]

with torch.no_grad():
    # TODO 11 — compute logits
    # logits = ...

    # TODO 12 — predicted classes
    # y_pred = ...

    # TODO 13 — probabilities, only because we want to inspect them
    # y_proba = ...

    pass  # remove after completing the TODOs

# print("true:", [class_names[i] for i in y_new.tolist()])
# print("pred:", [class_names[i] for i in y_pred.cpu().tolist()])
# print("confidence:", y_proba.max(dim=1).values.cpu())


## 10. The mental model to leave on screen

A PyTorch classifier is not “magic”. The core is:

$
X
\xrightarrow{\text{model}}
\text{logits}
\xrightarrow{\text{loss}}
L
\xrightarrow{\text{backward}}
\nabla_\theta L
\xrightarrow{\text{optimizer.step}}
\theta'
$

**Five lines to remember:**

```python
logits = model(X)
loss = criterion(logits, y)
loss.backward()
optimizer.step()
optimizer.zero_grad()
```

### Optional 60-second questions

1. Why does the final layer have 10 units?
2. Why is there no Softmax inside the model?
3. What would happen if we forgot `optimizer.zero_grad()`?
4. Why do we call `model.eval()` for validation?
5. Why can accuracy be used for validation but not as our gradient-descent objective?

**Source:** Géron, Ch. 10, especially “Building an Image Classifier with PyTorch” and the chapter's mini-batch training loop.


## Optional extensions

These are deliberately **not** part of the 30-minute live path:

- set `FAST_MODE=False`;
- train for more epochs;
- compare SGD learning rates;
- inspect misclassified images;
- compare `CrossEntropyLoss(logits, y)` with `LogSoftmax + NLLLoss`;
- try `BCEWithLogitsLoss` on a binary version of the task;
- add TorchMetrics;
- save and reload the model.

Géron reports that the Chapter 10 Fashion-MNIST MLP reaches roughly **92.8% training accuracy and 87.2% validation accuracy**, with variation depending on hardware and training details.
